In [1]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 01 — ENVIRONMENT AND INPUT VERIFICATION
# ============================================================

from google.colab import drive
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import json

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project")

BASELINE_MODEL_PATH = (
    PROJECT_ROOT / "models/RF_B_CIC/saved_model"
)

ENHANCED_MODEL_PATH = (
    PROJECT_ROOT / "models/RF_I1_CIC/RF_I1_S/saved_model"
)

TEST_PATH = (
    PROJECT_ROOT
    / "data/CICIDS2017/processed/cicids2017_test.parquet"
)

BASELINE_RESULTS = PROJECT_ROOT / "results/RF_B_CIC"
ENHANCED_RESULTS = PROJECT_ROOT / "results/RF_I1_CIC"
RF_E_RESULTS = PROJECT_ROOT / "results/RF_E_CIC"

COMPLETION_PATH = (
    RF_E_RESULTS / "RF_E_CIC_NOTEBOOK_07_COMPLETE.json"
)

print("=" * 72)
print("CELL 01 — ENVIRONMENT AND INPUT VERIFICATION")
print("=" * 72)

required_files = {
    "Baseline SavedModel":
        BASELINE_MODEL_PATH / "saved_model.pb",
    "Enhanced SavedModel":
        ENHANCED_MODEL_PATH / "saved_model.pb",
    "CIC test partition":
        TEST_PATH,
    "Baseline metrics":
        BASELINE_RESULTS / "RF_B_CIC_metrics.csv",
    "Enhanced metrics":
        ENHANCED_RESULTS / "RF_I1_S_metrics.csv",
    "RF-E completion record":
        COMPLETION_PATH,
}

checks = {}

for label, path in required_files.items():
    present = path.is_file()
    checks[label] = present
    print(f"{'PASS' if present else 'FAIL':<6} | {label}")

if not all(checks.values()):
    raise RuntimeError("Required input artefacts are missing.")

completion = json.loads(
    COMPLETION_PATH.read_text(encoding="utf-8")
)

if (
    completion.get("final_model") != "RF-E-CIC"
    or completion.get("source_model") != "RF-I1-S"
    or completion.get("status") != "COMPLETE"
):
    raise RuntimeError("Invalid RF-E completion record.")

print("\nSOFTWARE VERSIONS")

for name in [
    "tensorflow",
    "tensorflow-decision-forests",
    "tf-keras",
    "pandas",
    "pyarrow",
    "scikit-learn",
    "statsmodels",
]:
    try:
        installed = version(name)
    except PackageNotFoundError:
        installed = "NOT INSTALLED"

    print(f"{name:<30}: {installed}")

print("\nMODEL PATHS")
print("RF-B-CIC:", BASELINE_MODEL_PATH)
print("RF-E-CIC:", ENHANCED_MODEL_PATH)

print("\nCELL 01: PASSED")


Mounted at /content/drive
CELL 01 — ENVIRONMENT AND INPUT VERIFICATION
PASS   | Baseline SavedModel
PASS   | Enhanced SavedModel
PASS   | CIC test partition
PASS   | Baseline metrics
PASS   | Enhanced metrics
PASS   | RF-E completion record

SOFTWARE VERSIONS
tensorflow                    : 2.19.0
tensorflow-decision-forests   : 1.12.0
tf-keras                      : 2.19.0
pandas                        : 2.2.2
pyarrow                       : 18.1.0
scikit-learn                  : 1.6.1
statsmodels                   : 0.14.6

MODEL PATHS
RF-B-CIC: /content/drive/MyDrive/MSc_Project/models/RF_B_CIC/saved_model
RF-E-CIC: /content/drive/MyDrive/MSc_Project/models/RF_I1_CIC/RF_I1_S/saved_model

CELL 01: PASSED


In [2]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 02 — SAVEDMODEL INTERFACE INSPECTION
# ============================================================

import tensorflow as tf
import tensorflow_decision_forests as tfdf

print("=" * 72)
print("CELL 02 — SAVEDMODEL INTERFACE INSPECTION")
print("=" * 72)

print("TensorFlow:", tf.__version__)
print("TensorFlow Decision Forests:", tfdf.__version__)

# Load frozen models
print("\nLOADING BASELINE MODEL")
BASELINE_MODEL = tf.saved_model.load(
    str(BASELINE_MODEL_PATH)
)

print("LOADING ENHANCED MODEL")
ENHANCED_MODEL = tf.saved_model.load(
    str(ENHANCED_MODEL_PATH)
)

SAVED_MODELS = {
    "RF-B-CIC": BASELINE_MODEL,
    "RF-E-CIC": ENHANCED_MODEL,
}

MODEL_SIGNATURES = {}

for model_name, model in SAVED_MODELS.items():

    print("\n" + "-" * 72)
    print(model_name)
    print("-" * 72)

    signature_names = list(model.signatures.keys())

    print("Available signatures:", signature_names)

    if "serving_default" not in model.signatures:
        raise RuntimeError(
            f"{model_name}: serving_default signature not found."
        )

    signature = model.signatures["serving_default"]

    MODEL_SIGNATURES[model_name] = signature

    args, kwargs = signature.structured_input_signature

    print("\nPositional input count:", len(args))
    print("Named input count:", len(kwargs))

    if kwargs:
        print("\nFirst 10 input specifications:")

        for name in sorted(kwargs)[:10]:
            spec = kwargs[name]
            print(
                f"  {name}: "
                f"shape={spec.shape}, dtype={spec.dtype.name}"
            )

    else:
        print("\nPositional input specifications:")
        for spec in args[:3]:
            print(" ", spec)

    print("\nOutput specifications:")

    outputs = signature.structured_outputs

    if isinstance(outputs, dict):
        for name, spec in outputs.items():
            print(
                f"  {name}: "
                f"shape={spec.shape}, dtype={spec.dtype.name}"
            )
    else:
        print(outputs)

print("\n" + "=" * 72)
print("CELL 02: COMPLETED")
print("Models loaded for inference: YES")
print("Model training performed: NO")
print("=" * 72)


CELL 02 — SAVEDMODEL INTERFACE INSPECTION
TensorFlow: 2.19.0
TensorFlow Decision Forests: 1.12.0

LOADING BASELINE MODEL
LOADING ENHANCED MODEL

------------------------------------------------------------------------
RF-B-CIC
------------------------------------------------------------------------
Available signatures: ['serving_default']

Positional input count: 0
Named input count: 78

First 10 input specifications:
  ACK_Flag_Count: shape=(None,), dtype=float32
  Active_Max: shape=(None,), dtype=float32
  Active_Mean: shape=(None,), dtype=float32
  Active_Min: shape=(None,), dtype=float32
  Active_Std: shape=(None,), dtype=float32
  Average_Packet_Size: shape=(None,), dtype=float32
  Avg_Bwd_Segment_Size: shape=(None,), dtype=float32
  Avg_Fwd_Segment_Size: shape=(None,), dtype=float32
  Bwd_Avg_Bulk_Rate: shape=(None,), dtype=float32
  Bwd_Avg_Bytes_Bulk: shape=(None,), dtype=float32

Output specifications:
  output_1: shape=(None, 15), dtype=float32

-----------------------------

In [5]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 03 — LABEL ENCODING AND CLASS MAPPING VERIFICATION
# ============================================================

from pathlib import Path
import tensorflow_decision_forests as tfdf
import pandas as pd
import pyarrow.parquet as pq
import json
import os
import re

print("=" * 72)
print("CELL 03 — LABEL ENCODING AND CLASS MAPPING")
print("=" * 72)

test_parquet = pq.ParquetFile(TEST_PATH)

test_labels = (
    pq.read_table(TEST_PATH, columns=["Label"])
    .column("Label")
    .to_pandas()
    .astype(str)
)

test_classes = sorted(test_labels.unique())

print("\nTEST DATASET")
print("Records:", test_parquet.metadata.num_rows)
print("Predictors:", len(test_parquet.schema_arrow.names) - 1)
print("Classes:", len(test_classes))

assert test_parquet.metadata.num_rows == 378119
assert len(test_classes) == 15

MODEL_INSPECTORS = {}

for model_name, model_path in {
    "RF-B-CIC": BASELINE_MODEL_PATH,
    "RF-E-CIC": ENHANCED_MODEL_PATH,
}.items():

    assets = model_path / "assets"

    prefixes = [
        p.name.removesuffix("data_spec.pb")
        for p in assets.glob("*data_spec.pb")
    ]

    if len(prefixes) != 1:
        raise RuntimeError(
            f"{model_name}: invalid metadata prefix count."
        )

    inspector = tfdf.inspector.make_inspector(
        str(assets),
        file_prefix=prefixes[0]
    )

    MODEL_INSPECTORS[model_name] = inspector

    header = inspector.header
    dataspec = inspector.dataspec

    if callable(header):
        header = header()

    if callable(dataspec):
        dataspec = dataspec()

    label_column = dataspec.columns[header.label_col_idx]
    categorical = label_column.categorical

    print("\n" + "-" * 72)
    print(model_name)
    print("-" * 72)

    print("Model type:", inspector.model_type())
    print("Tree count:", inspector.num_trees())
    print("Objective:", inspector.objective())
    print("Label column:", label_column.name)
    print("Label type:", label_column.type)
    print(
        "Integer-encoded labels:",
        categorical.is_already_integerized
    )
    print(
        "Categorical values:",
        categorical.number_of_unique_values
    )

    classes = inspector.label_classes()

    print(
        "Stored string class labels:",
        "Unavailable" if classes is None else list(classes)
    )

# ------------------------------------------------------------
# Identify label-encoding definitions in project source files
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("PROJECT SOURCE CODE DISCOVERY")
print("=" * 72)

source_files = []
excluded_dirs = {
    "models", "results", "data",
    ".ipynb_checkpoints", "__pycache__"
}

for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [
        d for d in dirs if d not in excluded_dirs
    ]

    for filename in files:
        if filename.lower().endswith((".ipynb", ".py")):
            source_files.append(Path(root) / filename)

print("Source files found:", len(source_files))

pattern = re.compile(
    r"LabelEncoder|class_names|CLASS_NAMES|"
    r"class_order|CLASS_ORDER|label_mapping|"
    r"label_to_index|label_to_id|classes_",
    re.IGNORECASE
)

matches_found = 0

for path in sorted(source_files):
    try:
        if path.suffix == ".ipynb":
            notebook = json.loads(
                path.read_text(encoding="utf-8")
            )
            code_blocks = [
                "".join(cell.get("source", []))
                for cell in notebook.get("cells", [])
                if cell.get("cell_type") == "code"
            ]
        else:
            code_blocks = [
                path.read_text(encoding="utf-8")
            ]

        for block_index, block in enumerate(code_blocks):
            matching_lines = [
                line.strip()
                for line in block.splitlines()
                if pattern.search(line)
            ]

            if matching_lines:
                print("\nSource:", path.name)
                print("Code block:", block_index)

                for line in matching_lines[:8]:
                    print(" ", line[:200])

                matches_found += 1

                if matches_found >= 15:
                    break

        if matches_found >= 15:
            break

    except (OSError, ValueError, UnicodeError) as error:
        print("Source inspection skipped:", path.name)

print("\nSource blocks with mapping references:", matches_found)

print("\n" + "=" * 72)
print("CELL 03 — METADATA INSPECTION COMPLETE")
print("Class-index mapping: REQUIRES CONFIRMATION")
print("Model training performed: NO")
print("=" * 72)


CELL 03 — LABEL ENCODING AND CLASS MAPPING

TEST DATASET
Records: 378119
Predictors: 78
Classes: 15

------------------------------------------------------------------------
RF-B-CIC
------------------------------------------------------------------------
Model type: RANDOM_FOREST
Tree count: 100
Objective: Classification(label=__LABEL, class=None, num_classes=15)
Label column: __LABEL
Label type: 4
Integer-encoded labels: True
Categorical values: 16
Stored string class labels: Unavailable

------------------------------------------------------------------------
RF-E-CIC
------------------------------------------------------------------------
Model type: RANDOM_FOREST
Tree count: 100
Objective: Classification(label=__LABEL, class=None, num_classes=15)
Label column: __LABEL
Label type: 4
Integer-encoded labels: True
Categorical values: 16
Stored string class labels: Unavailable

PROJECT SOURCE CODE DISCOVERY
Source files found: 0

Source blocks with mapping references: 0

CELL 03 — META

In [6]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 03B — CLASS ORDER RECONSTRUCTION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

print("=" * 72)
print("CELL 03B — CLASS ORDER RECONSTRUCTION")
print("=" * 72)

report_path = (
    ENHANCED_RESULTS / "RF_I1_S_classification_report.csv"
)

matrix_path = (
    ENHANCED_RESULTS / "RF_I1_S_confusion_matrix.csv"
)

report = pd.read_csv(report_path)
report_classes = report.iloc[:15, 0].astype(str).tolist()

test_labels = (
    pq.read_table(TEST_PATH, columns=["Label"])
    .column("Label")
    .to_pandas()
    .astype(str)
)

dataset_classes = sorted(test_labels.unique().tolist())

if len(dataset_classes) != 15:
    raise RuntimeError("Unexpected number of test classes.")

if len(report_classes) != 15:
    raise RuntimeError("Unexpected classification report length.")

if set(report_classes) != set(dataset_classes):
    raise RuntimeError(
        "Classification report labels do not match test labels."
    )

print("\nCLASS ORDER COMPARISON")
print("Dataset labels:", len(dataset_classes))
print("Report labels:", len(report_classes))
print(
    "Report uses lexicographic order:",
    report_classes == dataset_classes
)

candidate_classes = dataset_classes

print("\nCANDIDATE OUTPUT MAPPING")

for index, class_name in enumerate(candidate_classes):
    print(f"{index:02d}: {class_name}")

# Verify the frozen confusion matrix

print("\nFROZEN CONFUSION MATRIX")

matrix_df = pd.read_csv(matrix_path, index_col=0)

matrix_df.index = matrix_df.index.astype(str).str.strip()
matrix_df.columns = [
    str(value).strip()
    for value in matrix_df.columns
]

if matrix_df.shape != (15, 15):
    raise RuntimeError(
        f"Unexpected confusion matrix shape: {matrix_df.shape}"
    )

if (
    set(matrix_df.index) != set(candidate_classes)
    or set(matrix_df.columns) != set(candidate_classes)
):
    raise RuntimeError(
        "Confusion matrix labels do not match dataset classes."
    )

matrix_df = matrix_df.loc[
    candidate_classes,
    candidate_classes
]

matrix_values = matrix_df.to_numpy(dtype=np.int64)

if np.any(matrix_values < 0):
    raise RuntimeError("Invalid negative confusion counts.")

expected_support = (
    test_labels.value_counts()
    .reindex(candidate_classes)
    .to_numpy(dtype=np.int64)
)

observed_support = matrix_values.sum(axis=1)

if not np.array_equal(
    observed_support,
    expected_support
):
    raise RuntimeError(
        "Confusion matrix class supports differ from test labels."
    )

if matrix_values.sum() != 378119:
    raise RuntimeError(
        "Confusion matrix total differs from test row count."
    )

MODEL_CLASS_ORDERS_CANDIDATE = {
    "RF-B-CIC": candidate_classes.copy(),
    "RF-E-CIC": candidate_classes.copy(),
}

RF08_CLASS_MAPPING_CONFIRMED = False

print("Matrix shape:", matrix_values.shape)
print("Matrix observations:", int(matrix_values.sum()))
print("Class supports:", "PASS")

print("\n" + "=" * 72)
print("CELL 03B: PASSED")
print("Candidate class order: ESTABLISHED")
print("Prediction-level mapping confirmation: PENDING")
print("=" * 72)


CELL 03B — CLASS ORDER RECONSTRUCTION

CLASS ORDER COMPARISON
Dataset labels: 15
Report labels: 15
Report uses lexicographic order: True

CANDIDATE OUTPUT MAPPING
00: BENIGN
01: Bot
02: DDoS
03: DoS GoldenEye
04: DoS Hulk
05: DoS Slowhttptest
06: DoS slowloris
07: FTP-Patator
08: Heartbleed
09: Infiltration
10: PortScan
11: SSH-Patator
12: Web Attack - Brute Force
13: Web Attack - SQL Injection
14: Web Attack - XSS

FROZEN CONFUSION MATRIX
Matrix shape: (15, 15)
Matrix observations: 378119
Class supports: PASS

CELL 03B: PASSED
Candidate class order: ESTABLISHED
Prediction-level mapping confirmation: PENDING


In [7]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 04 — STRATIFIED SAMPLE INFERENCE
# ============================================================

import re
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import tensorflow as tf

if "MODEL_CLASS_ORDERS_CANDIDATE" not in globals():
    raise RuntimeError("Cell 03B must pass first.")

print("=" * 72)
print("CELL 04 — STRATIFIED SAMPLE INFERENCE")
print("=" * 72)

# Select sample indices across all classes

sample_indices = []

label_array = test_labels.to_numpy()

for class_name in candidate_classes:
    positions = np.flatnonzero(label_array == class_name)
    sample_indices.extend(positions[:4].tolist())

sample_indices = np.asarray(
    sorted(sample_indices),
    dtype=np.int64
)

test_table = pq.read_table(TEST_PATH)

sample_df = (
    test_table.take(pa.array(sample_indices))
    .to_pandas()
    .reset_index(drop=True)
)

del test_table

sample_y = sample_df["Label"].astype(str).to_numpy()

print("Sample observations:", len(sample_df))
print("Sample classes:", len(np.unique(sample_y)))

def normalise_feature_name(value):
    value = str(value).strip()
    value = re.sub(r"[^A-Za-z0-9_]", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")

def resolve_feature_mapping(signature, dataframe):
    _, signature_inputs = signature.structured_input_signature

    dataset_features = [
        col for col in dataframe.columns
        if col != "Label"
    ]

    normalised = {}

    for col in dataset_features:
        key = normalise_feature_name(col)
        normalised.setdefault(key, []).append(col)

    mapping = {}

    for feature_name in signature_inputs:
        if feature_name in dataset_features:
            mapping[feature_name] = feature_name
            continue

        key = normalise_feature_name(feature_name)
        matches = normalised.get(key, [])

        if len(matches) != 1:
            raise RuntimeError(
                f"Feature mapping unresolved: {feature_name}"
            )

        mapping[feature_name] = matches[0]

    if len(mapping) != 78:
        raise RuntimeError("Expected 78 mapped input features.")

    if len(set(mapping.values())) != 78:
        raise RuntimeError("Duplicate source feature mapping.")

    return mapping

RF08_FEATURE_MAPS = {}
RF08_SAMPLE_PREDICTIONS = {}

for model_name in ["RF-B-CIC", "RF-E-CIC"]:

    signature = MODEL_SIGNATURES[model_name]

    feature_mapping = resolve_feature_mapping(
        signature,
        sample_df
    )

    RF08_FEATURE_MAPS[model_name] = feature_mapping

    inputs = {}

    for input_name, source_name in feature_mapping.items():
        values = pd.to_numeric(
            sample_df[source_name],
            errors="raise"
        ).to_numpy(dtype=np.float32)

        if not np.isfinite(values).all():
            raise RuntimeError(
                f"Non-finite feature values: {source_name}"
            )

        inputs[input_name] = tf.convert_to_tensor(
            values,
            dtype=tf.float32
        )

    outputs = signature(**inputs)

    scores = outputs["output_1"].numpy()

    if scores.shape != (len(sample_df), 15):
        raise RuntimeError(
            f"Invalid output shape: {scores.shape}"
        )

    if not np.isfinite(scores).all():
        raise RuntimeError("Non-finite prediction scores.")

    predicted_indices = np.argmax(scores, axis=1)

    classes = np.asarray(
        MODEL_CLASS_ORDERS_CANDIDATE[model_name],
        dtype=object
    )

    predictions = classes[predicted_indices]

    RF08_SAMPLE_PREDICTIONS[model_name] = predictions

    print("\n" + "-" * 72)
    print(model_name)
    print("-" * 72)
    print("Input features:", len(feature_mapping))
    print("Output shape:", scores.shape)
    print(
        "Correct sample predictions:",
        int(np.sum(predictions == sample_y)),
        "/",
        len(sample_y)
    )

# Class-level sample comparison

sample_summary = []

for class_name in candidate_classes:

    mask = sample_y == class_name

    sample_summary.append({
        "Class": class_name,
        "Support": int(mask.sum()),
        "Baseline Correct": int(
            np.sum(
                RF08_SAMPLE_PREDICTIONS["RF-B-CIC"][mask]
                == sample_y[mask]
            )
        ),
        "Enhanced Correct": int(
            np.sum(
                RF08_SAMPLE_PREDICTIONS["RF-E-CIC"][mask]
                == sample_y[mask]
            )
        ),
    })

sample_summary_df = pd.DataFrame(sample_summary)

print("\n" + "=" * 72)
print("CLASS-LEVEL SAMPLE RESULTS")
print("=" * 72)

print(sample_summary_df.to_string(index=False))

print("\n" + "=" * 72)
print("CELL 04: PASSED")
print("Sample inference completed: YES")
print("Full-test metric verification: PENDING")
print("Model training performed: NO")
print("=" * 72)


CELL 04 — STRATIFIED SAMPLE INFERENCE
Sample observations: 56
Sample classes: 15

------------------------------------------------------------------------
RF-B-CIC
------------------------------------------------------------------------
Input features: 78
Output shape: (56, 15)
Correct sample predictions: 50 / 56

------------------------------------------------------------------------
RF-E-CIC
------------------------------------------------------------------------
Input features: 78
Output shape: (56, 15)
Correct sample predictions: 52 / 56

CLASS-LEVEL SAMPLE RESULTS
                     Class  Support  Baseline Correct  Enhanced Correct
                    BENIGN        4                 4                 4
                       Bot        4                 3                 3
                      DDoS        4                 4                 4
             DoS GoldenEye        4                 4                 4
                  DoS Hulk        4                 4          

In [8]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 05 — BASELINE FULL-TEST INFERENCE
# ============================================================

import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import tensorflow as tf

print("=" * 72)
print("CELL 05 — BASELINE FULL-TEST INFERENCE")
print("=" * 72)

required_objects = [
    "MODEL_SIGNATURES",
    "MODEL_CLASS_ORDERS_CANDIDATE",
    "RF08_FEATURE_MAPS",
    "RF08_SAMPLE_PREDICTIONS",
    "sample_indices",
    "test_labels",
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required objects: " + ", ".join(missing)
    )

RF08_CLASSES = np.asarray(
    MODEL_CLASS_ORDERS_CANDIDATE["RF-B-CIC"],
    dtype=str
)

if len(RF08_CLASSES) != 15:
    raise RuntimeError("Expected 15 class labels.")

if RF08_CLASSES.tolist() != list(
    MODEL_CLASS_ORDERS_CANDIDATE["RF-E-CIC"]
):
    raise RuntimeError(
        "Candidate class orders differ between models."
    )

RF08_TRUE_LABELS = (
    test_labels.astype(str).to_numpy()
)

RF08_TOTAL_ROWS = len(RF08_TRUE_LABELS)

if RF08_TOTAL_ROWS != 378119:
    raise RuntimeError("Unexpected test observation count.")

RF08_LABEL_TO_INDEX = {
    label: index
    for index, label in enumerate(RF08_CLASSES)
}

RF08_TRUE_INDICES = np.asarray(
    [RF08_LABEL_TO_INDEX[label]
     for label in RF08_TRUE_LABELS],
    dtype=np.int16
)

def generate_full_predictions(model_name, batch_size=8192):

    signature = MODEL_SIGNATURES[model_name]
    feature_map = RF08_FEATURE_MAPS[model_name]

    if len(feature_map) != 78:
        raise RuntimeError(
            f"{model_name}: expected 78 input features."
        )

    _, signature_inputs = signature.structured_input_signature

    if set(feature_map) != set(signature_inputs):
        raise RuntimeError(
            f"{model_name}: feature mapping mismatch."
        )

    input_columns = list(
        dict.fromkeys(feature_map.values())
    )

    reader = pq.ParquetFile(TEST_PATH)

    if reader.metadata.num_rows != RF08_TOTAL_ROWS:
        raise RuntimeError("Parquet row count mismatch.")

    predicted_indices = np.empty(
        RF08_TOTAL_ROWS,
        dtype=np.int16
    )

    offset = 0
    start_time = time.perf_counter()

    for batch_number, batch in enumerate(
        reader.iter_batches(
            batch_size=batch_size,
            columns=input_columns + ["Label"]
        ),
        start=1
    ):
        frame = batch.to_pandas()
        batch_rows = len(frame)
        end = offset + batch_rows

        actual_labels = (
            frame["Label"].astype(str).to_numpy()
        )

        if not np.array_equal(
            actual_labels,
            RF08_TRUE_LABELS[offset:end]
        ):
            raise RuntimeError(
                f"{model_name}: test row alignment mismatch "
                f"at offset {offset}."
            )

        model_inputs = {}

        for model_feature, source_feature in feature_map.items():

            values = pd.to_numeric(
                frame[source_feature],
                errors="raise"
            ).to_numpy(dtype=np.float32)

            if not np.isfinite(values).all():
                raise RuntimeError(
                    f"{model_name}: non-finite values in "
                    f"{source_feature}."
                )

            model_inputs[model_feature] = (
                tf.convert_to_tensor(
                    values,
                    dtype=tf.float32
                )
            )

        outputs = signature(**model_inputs)

        if "output_1" not in outputs:
            raise RuntimeError(
                f"{model_name}: output_1 unavailable."
            )

        scores = outputs["output_1"].numpy()

        if scores.shape != (batch_rows, 15):
            raise RuntimeError(
                f"{model_name}: invalid output shape "
                f"{scores.shape}."
            )

        if not np.isfinite(scores).all():
            raise RuntimeError(
                f"{model_name}: non-finite prediction scores."
            )

        predicted_indices[offset:end] = (
            np.argmax(scores, axis=1).astype(np.int16)
        )

        offset = end

        if batch_number % 10 == 0:
            print(
                f"{model_name}: processed "
                f"{offset:,}/{RF08_TOTAL_ROWS:,} records"
            )

    if offset != RF08_TOTAL_ROWS:
        raise RuntimeError(
            f"{model_name}: incomplete test predictions."
        )

    predicted_labels = RF08_CLASSES[predicted_indices]

    sample_check = np.array_equal(
        predicted_labels[sample_indices],
        RF08_SAMPLE_PREDICTIONS[model_name]
    )

    if not sample_check:
        raise RuntimeError(
            f"{model_name}: sample predictions differ "
            "from Cell 04."
        )

    elapsed = time.perf_counter() - start_time

    return {
        "indices": predicted_indices,
        "elapsed_seconds": elapsed,
        "sample_verified": sample_check
    }

RF08_BASELINE = generate_full_predictions("RF-B-CIC")

baseline_correct = int(
    np.sum(
        RF08_BASELINE["indices"] == RF08_TRUE_INDICES
    )
)

print("\n" + "=" * 72)
print("BASELINE INFERENCE SUMMARY")
print("=" * 72)

print("Model: RF-B-CIC")
print("Test observations:", RF08_TOTAL_ROWS)
print("Prediction observations:", len(RF08_BASELINE["indices"]))
print("Prediction classes:", len(RF08_CLASSES))
print("Correct predictions:", baseline_correct)
print("Sample consistency:", RF08_BASELINE["sample_verified"])
print(
    "Inference time (seconds):",
    round(RF08_BASELINE["elapsed_seconds"], 3)
)
print("CELL 05: PASSED")
print("Model training performed: NO")
print("=" * 72)


CELL 05 — BASELINE FULL-TEST INFERENCE
RF-B-CIC: processed 81,920/378,119 records
RF-B-CIC: processed 163,840/378,119 records
RF-B-CIC: processed 245,760/378,119 records
RF-B-CIC: processed 327,680/378,119 records

BASELINE INFERENCE SUMMARY
Model: RF-B-CIC
Test observations: 378119
Prediction observations: 378119
Prediction classes: 15
Correct predictions: 377592
Sample consistency: True
Inference time (seconds): 7.947
CELL 05: PASSED
Model training performed: NO


In [9]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 06 — ENHANCED FULL-TEST INFERENCE
# ============================================================

import numpy as np

print("=" * 72)
print("CELL 06 — ENHANCED FULL-TEST INFERENCE")
print("=" * 72)

if "RF08_BASELINE" not in globals():
    raise RuntimeError("Cell 05 must pass first.")

if len(RF08_BASELINE["indices"]) != 378119:
    raise RuntimeError(
        "Baseline prediction count is invalid."
    )

RF08_ENHANCED = generate_full_predictions("RF-E-CIC")

enhanced_correct = int(
    np.sum(
        RF08_ENHANCED["indices"] == RF08_TRUE_INDICES
    )
)

print("\n" + "=" * 72)
print("ENHANCED INFERENCE SUMMARY")
print("=" * 72)

print("Model: RF-E-CIC")
print("Source: RF-I1-S")
print("Test observations:", RF08_TOTAL_ROWS)
print("Prediction observations:", len(RF08_ENHANCED["indices"]))
print("Prediction classes:", len(RF08_CLASSES))
print("Correct predictions:", enhanced_correct)
print("Sample consistency:", RF08_ENHANCED["sample_verified"])
print(
    "Inference time (seconds):",
    round(RF08_ENHANCED["elapsed_seconds"], 3)
)

print("\nPAIRED DATA ALIGNMENT")
print(
    "Baseline predictions:",
    len(RF08_BASELINE["indices"])
)
print(
    "Enhanced predictions:",
    len(RF08_ENHANCED["indices"])
)
print("True labels:", len(RF08_TRUE_INDICES))

if not (
    len(RF08_BASELINE["indices"])
    == len(RF08_ENHANCED["indices"])
    == len(RF08_TRUE_INDICES)
):
    raise RuntimeError("Paired prediction lengths differ.")

print("Paired observation counts: PASS")
print("CELL 06: PASSED")
print("Model training performed: NO")
print("=" * 72)


CELL 06 — ENHANCED FULL-TEST INFERENCE
RF-E-CIC: processed 81,920/378,119 records
RF-E-CIC: processed 163,840/378,119 records
RF-E-CIC: processed 245,760/378,119 records
RF-E-CIC: processed 327,680/378,119 records

ENHANCED INFERENCE SUMMARY
Model: RF-E-CIC
Source: RF-I1-S
Test observations: 378119
Prediction observations: 378119
Prediction classes: 15
Correct predictions: 377602
Sample consistency: True
Inference time (seconds): 8.049

PAIRED DATA ALIGNMENT
Baseline predictions: 378119
Enhanced predictions: 378119
True labels: 378119
Paired observation counts: PASS
CELL 06: PASSED
Model training performed: NO


In [10]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 07 — PREDICTION AND METRIC VERIFICATION
# ============================================================

from pathlib import Path
import io
import json
import hashlib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

print("=" * 72)
print("CELL 07 — PREDICTION AND METRIC VERIFICATION")
print("=" * 72)

if (
    "RF08_BASELINE" not in globals()
    or "RF08_ENHANCED" not in globals()
):
    raise RuntimeError(
        "Complete Cells 05 and 06 first."
    )

RESULT_SOURCES = {
    "RF-B-CIC": {
        "predictions": RF08_BASELINE["indices"],
        "metrics": (
            BASELINE_RESULTS / "RF_B_CIC_metrics.csv"
        ),
        "report": (
            BASELINE_RESULTS
            / "RF_B_CIC_classification_report.csv"
        )
    },
    "RF-E-CIC": {
        "predictions": RF08_ENHANCED["indices"],
        "metrics": (
            ENHANCED_RESULTS / "RF_I1_S_metrics.csv"
        ),
        "report": (
            ENHANCED_RESULTS
            / "RF_I1_S_classification_report.csv"
        )
    }
}

METRIC_NAMES = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted F1",
    "Balanced Accuracy",
    "MCC"
]

RF08_VERIFIED_METRICS = {}
RF08_CONFUSION_MATRICES = {}

def compute_metrics(y_true, y_pred):

    labels = list(range(15))

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true, y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true, y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        ),
        "Weighted F1": f1_score(
            y_true, y_pred,
            labels=labels,
            average="weighted",
            zero_division=0
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred)
    }

def verify_classification_report(
    source_path,
    y_true,
    y_pred
):
    archived = pd.read_csv(source_path)

    if len(archived) < 15:
        raise RuntimeError(
            "Archived classification report is incomplete."
        )

    archived_classes = (
        archived.iloc[:15, 0].astype(str).tolist()
    )

    if set(archived_classes) != set(RF08_CLASSES):
        raise RuntimeError(
            "Archived classification report class mismatch."
        )

    computed = classification_report(
        y_true,
        y_pred,
        labels=list(range(15)),
        target_names=RF08_CLASSES.tolist(),
        output_dict=True,
        zero_division=0
    )

    class_column = archived.columns[0]
    archived = archived.set_index(class_column)

    for class_name in RF08_CLASSES:

        row = archived.loc[class_name]
        reference = computed[class_name]

        for metric in [
            "precision", "recall", "f1-score", "support"
        ]:
            actual = float(reference[metric])
            expected = float(row[metric])

            tolerance = (
                0 if metric == "support"
                else 0.000002
            )

            if not np.isclose(
                actual,
                expected,
                rtol=0,
                atol=tolerance
            ):
                raise RuntimeError(
                    f"Classification report mismatch: "
                    f"{class_name}, {metric}. "
                    f"Computed={actual:.9f}, "
                    f"Archived={expected:.9f}"
                )

for model_name, source in RESULT_SOURCES.items():

    print("\n" + "-" * 72)
    print(model_name)
    print("-" * 72)

    y_pred = source["predictions"]

    computed = compute_metrics(
        RF08_TRUE_INDICES, y_pred
    )

    archived_df = pd.read_csv(source["metrics"])

    if len(archived_df) != 1:
        raise RuntimeError(
            f"{model_name}: expected one archived metrics row."
        )

    archived = archived_df.iloc[0]

    print(
        f"{'Metric':<22}"
        f"{'Computed':>14}"
        f"{'Archived':>14}"
        f"{'Status':>10}"
    )

    for metric in METRIC_NAMES:

        if metric not in archived.index:
            if metric == "Balanced Accuracy":
                archived_value = float(
                    archived["Macro Recall"]
                )
            else:
                raise RuntimeError(
                    f"{model_name}: missing metric {metric}."
                )
        else:
            archived_value = float(archived[metric])

        computed_value = float(computed[metric])

        passed = np.isclose(
            computed_value,
            archived_value,
            rtol=0,
            atol=0.000002
        )

        print(
            f"{metric:<22}"
            f"{computed_value:>14.6f}"
            f"{archived_value:>14.6f}"
            f"{'PASS' if passed else 'FAIL':>10}"
        )

        if not passed:
            raise RuntimeError(
                f"{model_name}: archived {metric} "
                "does not match reconstructed predictions."
            )

    verify_classification_report(
        source["report"],
        RF08_TRUE_INDICES,
        y_pred
    )

    print("Per-class classification report: PASS")

    cm = confusion_matrix(
        RF08_TRUE_INDICES,
        y_pred,
        labels=list(range(15))
    )

    RF08_CONFUSION_MATRICES[model_name] = cm
    RF08_VERIFIED_METRICS[model_name] = computed

# Verify the enhanced model's archived confusion matrix

print("\n" + "=" * 72)
print("ENHANCED CONFUSION MATRIX VERIFICATION")
print("=" * 72)

archived_cm_path = (
    ENHANCED_RESULTS / "RF_I1_S_confusion_matrix.csv"
)

archived_cm_df = pd.read_csv(
    archived_cm_path,
    index_col=0
)

archived_cm_df.index = (
    archived_cm_df.index.astype(str).str.strip()
)

archived_cm_df.columns = [
    str(column).strip()
    for column in archived_cm_df.columns
]

if (
    set(archived_cm_df.index) != set(RF08_CLASSES)
    or set(archived_cm_df.columns) != set(RF08_CLASSES)
):
    raise RuntimeError(
        "Archived confusion matrix class labels differ."
    )

archived_cm_df = archived_cm_df.loc[
    RF08_CLASSES.tolist(),
    RF08_CLASSES.tolist()
]

archived_cm = archived_cm_df.to_numpy(
    dtype=np.int64
)

computed_cm = RF08_CONFUSION_MATRICES["RF-E-CIC"]

if not np.array_equal(archived_cm, computed_cm):
    differing_cells = int(
        np.sum(archived_cm != computed_cm)
    )
    maximum_difference = int(
        np.max(np.abs(archived_cm - computed_cm))
    )

    raise RuntimeError(
        "Enhanced confusion matrix mismatch. "
        f"Differing cells: {differing_cells}; "
        f"maximum difference: {maximum_difference}."
    )

print("Archived matrix shape:", archived_cm.shape)
print("Matching matrix cells:", archived_cm.size)
print("Exact confusion matrix agreement: PASS")

RF08_CLASS_MAPPING_CONFIRMED = True

# Save verified prediction indices

OUTPUT_DIR_08 = (
    PROJECT_ROOT / "results/RF_STATS_CIC"
)

OUTPUT_DIR_08.mkdir(parents=True, exist_ok=True)

def write_bytes_once(path, content):

    path = Path(path)

    try:
        with path.open("xb") as handle:
            handle.write(content)
        return "CREATED"

    except FileExistsError:
        if path.read_bytes() == content:
            return "ALREADY VERIFIED"

        raise RuntimeError(
            f"Existing artefact differs: {path}"
        )

def save_array_once(path, array):

    buffer = io.BytesIO()

    np.save(
        buffer,
        np.asarray(array, dtype=np.int16),
        allow_pickle=False
    )

    return write_bytes_once(
        path,
        buffer.getvalue()
    )

saved_arrays = {
    "RF08_test_true_indices.npy": RF08_TRUE_INDICES,
    "RF08_baseline_pred_indices.npy": (
        RF08_BASELINE["indices"]
    ),
    "RF08_enhanced_pred_indices.npy": (
        RF08_ENHANCED["indices"]
    ),
}

print("\n" + "=" * 72)
print("VERIFIED PREDICTION ARTEFACTS")
print("=" * 72)

for filename, array in saved_arrays.items():

    path = OUTPUT_DIR_08 / filename

    status = save_array_once(path, array)

    print(f"{filename}: {status}")

class_mapping = {
    "dataset": "CIC-IDS2017",
    "models": ["RF-B-CIC", "RF-E-CIC"],
    "class_names_by_output_index": (
        RF08_CLASSES.tolist()
    ),
    "class_count": 15,
    "test_observations": 378119,
    "mapping_verified_against": [
        "RF-B-CIC archived metrics",
        "RF-E-CIC archived metrics",
        "Both archived per-class classification reports",
        "RF-I1-S archived confusion matrix"
    ]
}

mapping_bytes = (
    json.dumps(
        class_mapping,
        indent=2,
        sort_keys=True
    ) + "\n"
).encode("utf-8")

mapping_path = (
    OUTPUT_DIR_08 / "RF08_class_mapping.json"
)

print(
    "RF08_class_mapping.json:",
    write_bytes_once(mapping_path, mapping_bytes)
)

metrics_rows = []

for model_name, values in RF08_VERIFIED_METRICS.items():
    metrics_rows.append({
        "Model": model_name,
        **values
    })

verification_df = pd.DataFrame(metrics_rows)

metrics_path = (
    OUTPUT_DIR_08 / "RF08_verified_test_metrics.csv"
)

metrics_bytes = verification_df.to_csv(
    index=False,
    float_format="%.9f"
).encode("utf-8")

print(
    "RF08_verified_test_metrics.csv:",
    write_bytes_once(metrics_path, metrics_bytes)
)

print("\n" + "=" * 72)
print("CELL 07 — VERIFICATION SUMMARY")
print("=" * 72)

print("Archived aggregate metrics: PASS")
print("Archived per-class reports: PASS")
print("Enhanced confusion matrix: PASS")
print("Class-index mapping: VERIFIED")
print("Paired prediction arrays: SAVED")
print("Test observations:", RF08_TOTAL_ROWS)
print("Output directory:", OUTPUT_DIR_08)
print("CELL 07: PASSED")
print("Model training performed: NO")
print("=" * 72)


CELL 07 — PREDICTION AND METRIC VERIFICATION

------------------------------------------------------------------------
RF-B-CIC
------------------------------------------------------------------------
Metric                      Computed      Archived    Status
Accuracy                    0.998606      0.998606      PASS
Macro Precision             0.935485      0.935485      PASS
Macro Recall                0.879297      0.879297      PASS
Macro F1                    0.893851      0.893851      PASS
Weighted F1                 0.998570      0.998570      PASS
Balanced Accuracy           0.879297      0.879297      PASS
MCC                         0.995364      0.995364      PASS
Per-class classification report: PASS

------------------------------------------------------------------------
RF-E-CIC
------------------------------------------------------------------------
Metric                      Computed      Archived    Status
Accuracy                    0.998633      0.998633      

In [12]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 08 — PAIRED CORRECTNESS CONTINGENCY
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project")
OUTPUT_DIR_08 = PROJECT_ROOT / "results/RF_STATS_CIC"

print("=" * 72)
print("CELL 08 — PAIRED CORRECTNESS CONTINGENCY")
print("=" * 72)

# Load verified prediction arrays

array_files = {
    "true": "RF08_test_true_indices.npy",
    "baseline": "RF08_baseline_pred_indices.npy",
    "enhanced": "RF08_enhanced_pred_indices.npy",
}

arrays = {}

for key, filename in array_files.items():
    path = OUTPUT_DIR_08 / filename

    if not path.is_file():
        raise FileNotFoundError(path)

    arrays[key] = np.load(
        path,
        mmap_mode="r",
        allow_pickle=False
    )

y_true = arrays["true"]
y_baseline = arrays["baseline"]
y_enhanced = arrays["enhanced"]

N = 378119

# Verify prediction dimensions and encoded classes

for name, array in arrays.items():

    if array.shape != (N,):
        raise RuntimeError(
            f"{name}: unexpected shape {array.shape}"
        )

    if not np.issubdtype(array.dtype, np.integer):
        raise RuntimeError(
            f"{name}: expected integer class indices."
        )

    if np.any(array < 0) or np.any(array >= 15):
        raise RuntimeError(
            f"{name}: invalid class indices."
        )

# Verify class mapping and archived metrics

mapping_path = OUTPUT_DIR_08 / "RF08_class_mapping.json"
metrics_path = OUTPUT_DIR_08 / "RF08_verified_test_metrics.csv"

class_mapping = json.loads(
    mapping_path.read_text(encoding="utf-8")
)

verified_metrics = pd.read_csv(metrics_path)

if len(class_mapping["class_names_by_output_index"]) != 15:
    raise RuntimeError("Invalid class mapping.")

# Calculate prediction correctness

baseline_correct = y_baseline == y_true
enhanced_correct = y_enhanced == y_true

baseline_correct_count = int(
    np.count_nonzero(baseline_correct)
)

enhanced_correct_count = int(
    np.count_nonzero(enhanced_correct)
)

assert baseline_correct_count == 377592
assert enhanced_correct_count == 377602

for model_name, correct_count in [
    ("RF-B-CIC", baseline_correct_count),
    ("RF-E-CIC", enhanced_correct_count),
]:

    record = verified_metrics.loc[
        verified_metrics["Model"] == model_name
    ]

    if len(record) != 1:
        raise RuntimeError(
            f"{model_name}: metrics record unavailable."
        )

    archived_accuracy = float(record.iloc[0]["Accuracy"])
    computed_accuracy = correct_count / N

    if not np.isclose(
        computed_accuracy,
        archived_accuracy,
        rtol=0,
        atol=1e-9
    ):
        raise RuntimeError(
            f"{model_name}: accuracy mismatch."
        )

# Construct paired correctness contingency

a = int(np.count_nonzero(
    baseline_correct & enhanced_correct
))

b = int(np.count_nonzero(
    baseline_correct & ~enhanced_correct
))

c = int(np.count_nonzero(
    ~baseline_correct & enhanced_correct
))

d = int(np.count_nonzero(
    ~baseline_correct & ~enhanced_correct
))

# Verify contingency identities

assert a + b + c + d == N

assert a + b == baseline_correct_count
assert a + c == enhanced_correct_count

assert c + d == N - baseline_correct_count
assert b + d == N - enhanced_correct_count

assert c - b == (
    enhanced_correct_count - baseline_correct_count
)

assert c - b == 10

RF08_PAIRED_COUNTS = {
    "a": a,
    "b": b,
    "c": c,
    "d": d,
    "n": N,
}

RF08_CONTINGENCY_VERIFIED = False

contingency = pd.DataFrame(
    [[a, b], [c, d]],
    index=["RF-B correct", "RF-B incorrect"],
    columns=["RF-E correct", "RF-E incorrect"]
)

print("\nPAIRED CONTINGENCY TABLE")
print(contingency.to_string())

print("\nPAIRWISE ERROR SUMMARY")
print("Total observations:", N)
print("Both correct (a):", a)
print("Baseline only correct (b):", b)
print("Enhanced only correct (c):", c)
print("Both incorrect (d):", d)
print("Discordant pairs:", b + c)
print("Net correctness improvement:", c - b)

print("\nCORRECTNESS VERIFICATION")
print("Baseline correct:", a + b)
print("Baseline incorrect:", c + d)
print("Enhanced correct:", a + c)
print("Enhanced incorrect:", b + d)
print("Contingency arithmetic: PASS")

# Export verified results

def write_bytes_once(path, content):
    path = Path(path)

    try:
        with path.open("xb") as handle:
            handle.write(content)
        return "CREATED"

    except FileExistsError:
        if path.read_bytes() == content:
            return "ALREADY VERIFIED"

        raise RuntimeError(
            f"Existing output differs: {path}"
        )

contingency_path = (
    OUTPUT_DIR_08 / "RF08_paired_contingency.csv"
)

counts_path = (
    OUTPUT_DIR_08 / "RF08_paired_counts.json"
)

csv_status = write_bytes_once(
    contingency_path,
    contingency.to_csv(
        index_label="Baseline status"
    ).encode("utf-8")
)

json_status = write_bytes_once(
    counts_path,
    (
        json.dumps(
            RF08_PAIRED_COUNTS,
            indent=2,
            sort_keys=True
        ) + "\n"
    ).encode("utf-8")
)

RF08_CONTINGENCY_VERIFIED = True

print("\nOUTPUT VERIFICATION")
print("Contingency CSV:", csv_status)
print("Paired counts JSON:", json_status)
print("Paired observation alignment: PASS")
print("CELL 08: PASSED")
print("Model training performed: NO")
print("=" * 72)


CELL 08 — PAIRED CORRECTNESS CONTINGENCY

PAIRED CONTINGENCY TABLE
                RF-E correct  RF-E incorrect
RF-B correct          377490             102
RF-B incorrect           112             415

PAIRWISE ERROR SUMMARY
Total observations: 378119
Both correct (a): 377490
Baseline only correct (b): 102
Enhanced only correct (c): 112
Both incorrect (d): 415
Discordant pairs: 214
Net correctness improvement: 10

CORRECTNESS VERIFICATION
Baseline correct: 377592
Baseline incorrect: 527
Enhanced correct: 377602
Enhanced incorrect: 517
Contingency arithmetic: PASS

OUTPUT VERIFICATION
Contingency CSV: CREATED
Paired counts JSON: CREATED
Paired observation alignment: PASS
CELL 08: PASSED
Model training performed: NO


In [13]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 09 — MCNEMAR STATISTICAL TEST
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.stats.contingency_tables import mcnemar

print("=" * 72)
print("CELL 09 — MCNEMAR STATISTICAL TEST")
print("=" * 72)

if not globals().get("RF08_CONTINGENCY_VERIFIED", False):
    raise RuntimeError("Cell 08 must pass first.")

counts_path = OUTPUT_DIR_08 / "RF08_paired_counts.json"

counts = json.loads(
    counts_path.read_text(encoding="utf-8")
)

a = int(counts["a"])
b = int(counts["b"])
c = int(counts["c"])
d = int(counts["d"])
n = int(counts["n"])

if a + b + c + d != n or n != 378119:
    raise RuntimeError("Invalid contingency counts.")

if c - b != 10:
    raise RuntimeError(
        "Paired correctness difference is inconsistent."
    )

contingency_table = np.array(
    [[a, b], [c, d]],
    dtype=np.int64
)

discordant = b + c
alpha = 0.05

if discordant == 0:
    test_method = "No discordant pairs"
    statistic = 0.0
    p_value = 1.0

else:
    use_exact = discordant <= 25

    test_result = mcnemar(
        contingency_table,
        exact=use_exact,
        correction=not use_exact
    )

    test_method = (
        "Exact binomial McNemar"
        if use_exact
        else "Continuity-corrected McNemar chi-square"
    )

    statistic = float(test_result.statistic)
    p_value = float(test_result.pvalue)

# Paired accuracy difference

baseline_accuracy = (a + b) / n
enhanced_accuracy = (a + c) / n

accuracy_difference = enhanced_accuracy - baseline_accuracy

if not np.isclose(
    accuracy_difference,
    (c - b) / n,
    rtol=0,
    atol=1e-12
):
    raise RuntimeError(
        "Paired accuracy difference verification failed."
    )

# Approximate paired 95% confidence interval

paired_variance = (
    (b + c) / n - ((c - b) / n) ** 2
) / n

standard_error = np.sqrt(
    max(paired_variance, 0.0)
)

z_critical = 1.959963984540054

ci_lower = accuracy_difference - z_critical * standard_error
ci_upper = accuracy_difference + z_critical * standard_error

# Discordant-pair odds ratio

if b == 0:
    discordant_odds_ratio = (
        float("inf") if c > 0 else float("nan")
    )
else:
    discordant_odds_ratio = c / b

significant = p_value < alpha

print("\nPAIRED CONTINGENCY COUNTS")
print("Both correct (a):", a)
print("Baseline only correct (b):", b)
print("Enhanced only correct (c):", c)
print("Both incorrect (d):", d)
print("Discordant pairs:", discordant)

print("\nMCNEMAR TEST")
print("Method:", test_method)
print("Statistic:", f"{statistic:.6f}")
print("Two-sided p-value:", f"{p_value:.8g}")
print("Significance threshold:", alpha)
print(
    "Decision:",
    "Reject equal marginal error rates"
    if significant
    else "Do not reject equal marginal error rates"
)

print("\nPAIRED EFFECT ESTIMATES")
print("Baseline accuracy:", f"{baseline_accuracy:.9f}")
print("Enhanced accuracy:", f"{enhanced_accuracy:.9f}")
print(
    "Accuracy difference:",
    f"{accuracy_difference:+.9f}"
)
print(
    "Accuracy change (percentage points):",
    f"{100 * accuracy_difference:+.6f}"
)

print(
    "Approximate 95% CI for accuracy difference:",
    f"[{ci_lower:+.9f}, {ci_upper:+.9f}]"
)

print(
    "Discordant-pair odds ratio (c/b):",
    f"{discordant_odds_ratio:.6f}"
)

# Statistical results record

RF08_MCNEMAR_RESULT = {
    "analysis": "Paired McNemar test",
    "dataset": "CIC-IDS2017",
    "model_baseline": "RF-B-CIC",
    "model_enhanced": "RF-E-CIC",
    "test_observations": n,
    "both_correct_a": a,
    "baseline_only_correct_b": b,
    "enhanced_only_correct_c": c,
    "both_incorrect_d": d,
    "discordant_pairs": discordant,
    "method": test_method,
    "test_statistic": statistic,
    "p_value_two_sided": p_value,
    "alpha": alpha,
    "reject_equal_marginal_error_rates": bool(significant),
    "baseline_accuracy": baseline_accuracy,
    "enhanced_accuracy": enhanced_accuracy,
    "accuracy_difference": accuracy_difference,
    "accuracy_difference_ci95_approximate": {
        "lower": float(ci_lower),
        "upper": float(ci_upper)
    },
    "discordant_odds_ratio": (
        discordant_odds_ratio
        if np.isfinite(discordant_odds_ratio)
        else None
    ),
    "metric_scope": "Paired overall classification correctness",
    "model_training_performed": False
}

results_json_path = (
    OUTPUT_DIR_08 / "RF08_McNemar_results.json"
)

json_content = (
    json.dumps(
        RF08_MCNEMAR_RESULT,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

json_status = write_bytes_once(
    results_json_path,
    json_content
)

results_csv_path = (
    OUTPUT_DIR_08 / "RF08_McNemar_results.csv"
)

flat_result = {
    key: value
    for key, value in RF08_MCNEMAR_RESULT.items()
    if not isinstance(value, dict)
}

flat_result["ci95_lower"] = float(ci_lower)
flat_result["ci95_upper"] = float(ci_upper)

csv_content = pd.DataFrame(
    [flat_result]
).to_csv(
    index=False,
    float_format="%.12g"
).encode("utf-8")

csv_status = write_bytes_once(
    results_csv_path,
    csv_content
)

RF08_MCNEMAR_VERIFIED = True

print("\nOUTPUT VERIFICATION")
print("McNemar JSON:", json_status)
print("McNemar CSV:", csv_status)
print("Paired test completed: YES")
print("Cell 09: PASSED")
print("Model training performed: NO")
print("=" * 72)


CELL 09 — MCNEMAR STATISTICAL TEST

PAIRED CONTINGENCY COUNTS
Both correct (a): 377490
Baseline only correct (b): 102
Enhanced only correct (c): 112
Both incorrect (d): 415
Discordant pairs: 214

MCNEMAR TEST
Method: Continuity-corrected McNemar chi-square
Statistic: 0.378505
Two-sided p-value: 0.5384046
Significance threshold: 0.05
Decision: Do not reject equal marginal error rates

PAIRED EFFECT ESTIMATES
Baseline accuracy: 0.998606259
Enhanced accuracy: 0.998632706
Accuracy difference: +0.000026447
Accuracy change (percentage points): +0.002645
Approximate 95% CI for accuracy difference: [-0.000049381, +0.000102274]
Discordant-pair odds ratio (c/b): 1.098039

OUTPUT VERIFICATION
McNemar JSON: CREATED
McNemar CSV: CREATED
Paired test completed: YES
Cell 09: PASSED
Model training performed: NO


In [14]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 10 — PER-CLASS PERFORMANCE AND ERROR ANALYSIS
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix
)

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project")
OUTPUT_DIR_08 = PROJECT_ROOT / "results/RF_STATS_CIC"

print("=" * 72)
print("CELL 10 — PER-CLASS PERFORMANCE AND ERROR ANALYSIS")
print("=" * 72)

y_true = np.load(
    OUTPUT_DIR_08 / "RF08_test_true_indices.npy",
    allow_pickle=False
)

y_baseline = np.load(
    OUTPUT_DIR_08 / "RF08_baseline_pred_indices.npy",
    allow_pickle=False
)

y_enhanced = np.load(
    OUTPUT_DIR_08 / "RF08_enhanced_pred_indices.npy",
    allow_pickle=False
)

mapping = json.loads(
    (
        OUTPUT_DIR_08 / "RF08_class_mapping.json"
    ).read_text(encoding="utf-8")
)

paired_counts = json.loads(
    (
        OUTPUT_DIR_08 / "RF08_paired_counts.json"
    ).read_text(encoding="utf-8")
)

class_names = mapping["class_names_by_output_index"]
labels = np.arange(len(class_names))

N = len(y_true)

if N != 378119:
    raise RuntimeError("Unexpected test observation count.")

if not (
    len(y_baseline) == len(y_enhanced) == N
):
    raise RuntimeError("Prediction array length mismatch.")

if len(class_names) != 15:
    raise RuntimeError("Invalid class mapping.")

# Aggregate per-class classification metrics

def get_class_metrics(predictions):
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            predictions,
            labels=labels,
            zero_division=0
        )
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support
    }

baseline_metrics = get_class_metrics(y_baseline)
enhanced_metrics = get_class_metrics(y_enhanced)

baseline_correct = y_baseline == y_true
enhanced_correct = y_enhanced == y_true

class_records = []
paired_class_records = []

for index, class_name in enumerate(class_names):

    mask = y_true == index

    b = int(np.count_nonzero(
        mask & baseline_correct & ~enhanced_correct
    ))

    c = int(np.count_nonzero(
        mask & ~baseline_correct & enhanced_correct
    ))

    both_incorrect = int(np.count_nonzero(
        mask & ~baseline_correct & ~enhanced_correct
    ))

    both_correct = int(np.count_nonzero(
        mask & baseline_correct & enhanced_correct
    ))

    support = int(np.count_nonzero(mask))

    b_f1 = float(baseline_metrics["f1"][index])
    e_f1 = float(enhanced_metrics["f1"][index])

    class_records.append({
        "Class": class_name,
        "Support": support,
        "Baseline Precision": float(
            baseline_metrics["precision"][index]
        ),
        "Enhanced Precision": float(
            enhanced_metrics["precision"][index]
        ),
        "Precision Change": float(
            enhanced_metrics["precision"][index]
            - baseline_metrics["precision"][index]
        ),
        "Baseline Recall": float(
            baseline_metrics["recall"][index]
        ),
        "Enhanced Recall": float(
            enhanced_metrics["recall"][index]
        ),
        "Recall Change": float(
            enhanced_metrics["recall"][index]
            - baseline_metrics["recall"][index]
        ),
        "Baseline F1": b_f1,
        "Enhanced F1": e_f1,
        "F1 Change": e_f1 - b_f1,
        "Baseline Correct": both_correct + b,
        "Enhanced Correct": both_correct + c,
        "Net Correct Change": c - b
    })

    paired_class_records.append({
        "Class": class_name,
        "Support": support,
        "Both Correct": both_correct,
        "Baseline Only Correct": b,
        "Enhanced Only Correct": c,
        "Both Incorrect": both_incorrect,
        "Discordant": b + c,
        "Net Correct Change": c - b
    })

RF08_CLASS_COMPARISON = pd.DataFrame(class_records)
RF08_PAIRED_CLASS = pd.DataFrame(paired_class_records)

RF08_CM_BASELINE = confusion_matrix(
    y_true, y_baseline, labels=labels
)

RF08_CM_ENHANCED = confusion_matrix(
    y_true, y_enhanced, labels=labels
)

# Compare error patterns between models

error_records = []

for true_index, true_class in enumerate(class_names):
    for pred_index, predicted_class in enumerate(class_names):

        if true_index == pred_index:
            continue

        baseline_count = int(
            RF08_CM_BASELINE[true_index, pred_index]
        )

        enhanced_count = int(
            RF08_CM_ENHANCED[true_index, pred_index]
        )

        if baseline_count == 0 and enhanced_count == 0:
            continue

        error_records.append({
            "Actual Class": true_class,
            "Predicted Class": predicted_class,
            "Baseline Errors": baseline_count,
            "Enhanced Errors": enhanced_count,
            "Error Change": enhanced_count - baseline_count
        })

RF08_ERROR_PATTERNS = pd.DataFrame(
    error_records,
    columns=[
        "Actual Class",
        "Predicted Class",
        "Baseline Errors",
        "Enhanced Errors",
        "Error Change"
    ]
)

RF08_ERROR_PATTERNS = (
    RF08_ERROR_PATTERNS.sort_values(
        ["Enhanced Errors", "Baseline Errors"],
        ascending=False
    ).reset_index(drop=True)
)

# Integrity verification

assert RF08_CLASS_COMPARISON["Support"].sum() == N
assert RF08_PAIRED_CLASS["Both Correct"].sum() == paired_counts["a"]
assert RF08_PAIRED_CLASS["Baseline Only Correct"].sum() == paired_counts["b"]
assert RF08_PAIRED_CLASS["Enhanced Only Correct"].sum() == paired_counts["c"]
assert RF08_PAIRED_CLASS["Both Incorrect"].sum() == paired_counts["d"]
assert RF08_CLASS_COMPARISON["Net Correct Change"].sum() == 10

assert RF08_CM_BASELINE.sum() == N
assert RF08_CM_ENHANCED.sum() == N

RF08_MACRO_F1_BASELINE = float(
    RF08_CLASS_COMPARISON["Baseline F1"].mean()
)

RF08_MACRO_F1_ENHANCED = float(
    RF08_CLASS_COMPARISON["Enhanced F1"].mean()
)

assert np.isclose(
    RF08_MACRO_F1_BASELINE,
    0.893851,
    atol=0.000002,
    rtol=0
)

assert np.isclose(
    RF08_MACRO_F1_ENHANCED,
    0.912553,
    atol=0.000002,
    rtol=0
)

print("\nPER-CLASS PERFORMANCE")
print(
    RF08_CLASS_COMPARISON[
        [
            "Class",
            "Support",
            "Baseline F1",
            "Enhanced F1",
            "F1 Change",
            "Recall Change",
            "Net Correct Change"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nPER-CLASS PAIRED CORRECTNESS")
print(RF08_PAIRED_CLASS.to_string(index=False))

print("\nLARGEST ENHANCED ERROR PATTERNS")
print(
    RF08_ERROR_PATTERNS.head(10).to_string(
        index=False
    )
)

print("\nSUMMARY")
print("Baseline Macro F1:", f"{RF08_MACRO_F1_BASELINE:.6f}")
print("Enhanced Macro F1:", f"{RF08_MACRO_F1_ENHANCED:.6f}")
print(
    "Macro F1 change:",
    f"{RF08_MACRO_F1_ENHANCED - RF08_MACRO_F1_BASELINE:+.6f}"
)
print(
    "Classes with F1 improvement:",
    int((RF08_CLASS_COMPARISON["F1 Change"] > 1e-10).sum())
)
print(
    "Classes with F1 deterioration:",
    int((RF08_CLASS_COMPARISON["F1 Change"] < -1e-10).sum())
)
print("Per-class contingency totals: PASS")
print("Confusion matrix totals: PASS")
print("CELL 10: PASSED")
print("=" * 72)


CELL 10 — PER-CLASS PERFORMANCE AND ERROR ANALYSIS

PER-CLASS PERFORMANCE
                     Class  Support  Baseline F1  Enhanced F1  F1 Change  Recall Change  Net Correct Change
                    BENIGN   314258     0.999354     0.999405   0.000051      -0.000165                 -52
                       Bot      292     0.825455     0.833073   0.007619       0.136986                  40
                      DDoS    19202     0.999922     0.999948   0.000026       0.000052                   1
             DoS GoldenEye     1543     0.995772     0.995455  -0.000318       0.001296                   2
                  DoS Hulk    25927     0.998631     0.998766   0.000135       0.000231                   6
          DoS Slowhttptest      784     0.992376     0.993631   0.001254      -0.001276                  -1
             DoS slowloris      808     0.993804     0.995043   0.001239       0.001238                   1
               FTP-Patator      890     0.998312     0.998312 

In [15]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 11 — STATISTICAL RESULTS AND FIGURE EXPORT
# ============================================================

from pathlib import Path
import io
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "RF08_CLASS_COMPARISON" not in globals():
    raise RuntimeError("Cell 10 must pass first.")

OUTPUT_DIR_08 = (
    Path("/content/drive/MyDrive/MSc_Project")
    / "results/RF_STATS_CIC"
)

OUTPUT_DIR_08.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("CELL 11 — STATISTICAL RESULTS AND FIGURE EXPORT")
print("=" * 72)

def write_bytes_once(path, content):
    path = Path(path)

    try:
        with path.open("xb") as handle:
            handle.write(content)
        return "CREATED"

    except FileExistsError:
        if path.read_bytes() == content:
            return "ALREADY VERIFIED"

        raise RuntimeError(
            f"Existing artefact differs: {path}"
        )

def export_csv(filename, dataframe, index=False):
    content = dataframe.to_csv(
        index=index,
        float_format="%.9f"
    ).encode("utf-8")

    return write_bytes_once(
        OUTPUT_DIR_08 / filename,
        content
    )

print("\nTABLE EXPORTS")

tables = {
    "RF08_per_class_comparison.csv":
        RF08_CLASS_COMPARISON,
    "RF08_per_class_paired_counts.csv":
        RF08_PAIRED_CLASS,
    "RF08_error_pattern_comparison.csv":
        RF08_ERROR_PATTERNS,
}

for filename, dataframe in tables.items():
    print(filename, ":", export_csv(filename, dataframe))

cm_baseline_df = pd.DataFrame(
    RF08_CM_BASELINE,
    index=class_names,
    columns=class_names
)

cm_enhanced_df = pd.DataFrame(
    RF08_CM_ENHANCED,
    index=class_names,
    columns=class_names
)

print(
    "RF08_baseline_confusion_matrix.csv :",
    export_csv(
        "RF08_baseline_confusion_matrix.csv",
        cm_baseline_df,
        index=True
    )
)

print(
    "RF08_enhanced_confusion_matrix.csv :",
    export_csv(
        "RF08_enhanced_confusion_matrix.csv",
        cm_enhanced_df,
        index=True
    )
)

# Per-class F1 comparison figure

positions = np.arange(len(class_names))
bar_height = 0.38

fig, ax = plt.subplots(figsize=(12, 9))

ax.barh(
    positions - bar_height / 2,
    RF08_CLASS_COMPARISON["Baseline F1"],
    height=bar_height,
    label="RF-B-CIC"
)

ax.barh(
    positions + bar_height / 2,
    RF08_CLASS_COMPARISON["Enhanced F1"],
    height=bar_height,
    label="RF-E-CIC"
)

ax.set_yticks(positions)
ax.set_yticklabels(class_names, fontsize=9)
ax.invert_yaxis()
ax.set_xlim(0, 1.05)
ax.set_xlabel("F1-score")
ax.set_title("CIC-IDS2017: Per-Class F1 Comparison")
ax.legend()
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()

buffer = io.BytesIO()
fig.savefig(
    buffer,
    format="png",
    dpi=180,
    metadata={"Software": "Matplotlib"}
)
plt.close(fig)

f1_status = write_bytes_once(
    OUTPUT_DIR_08 / "RF08_per_class_f1_comparison.png",
    buffer.getvalue()
)

# Per-class paired correctness change

fig, ax = plt.subplots(figsize=(12, 9))

ax.barh(
    positions,
    RF08_CLASS_COMPARISON["Net Correct Change"]
)

ax.set_yticks(positions)
ax.set_yticklabels(class_names, fontsize=9)
ax.invert_yaxis()
ax.axvline(0, linewidth=1)
ax.set_xlabel("Change in correctly classified observations")
ax.set_title(
    "CIC-IDS2017: RF-E-CIC Versus RF-B-CIC"
)
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()

buffer = io.BytesIO()
fig.savefig(
    buffer,
    format="png",
    dpi=180,
    metadata={"Software": "Matplotlib"}
)
plt.close(fig)

change_status = write_bytes_once(
    OUTPUT_DIR_08 / "RF08_per_class_correct_change.png",
    buffer.getvalue()
)

print("\nFIGURE EXPORTS")
print("Per-class F1 comparison:", f1_status)
print("Per-class correctness change:", change_status)

# Consolidated analysis summary

mcnemar_result = json.loads(
    (
        OUTPUT_DIR_08 / "RF08_McNemar_results.json"
    ).read_text(encoding="utf-8")
)

statistical_summary = {
    "dataset": "CIC-IDS2017",
    "baseline_model": "RF-B-CIC",
    "enhanced_model": "RF-E-CIC",
    "observations": 378119,
    "classes": 15,
    "aggregate_metrics": {
        "baseline_macro_f1": RF08_MACRO_F1_BASELINE,
        "enhanced_macro_f1": RF08_MACRO_F1_ENHANCED,
        "macro_f1_difference": (
            RF08_MACRO_F1_ENHANCED
            - RF08_MACRO_F1_BASELINE
        )
    },
    "paired_correctness": {
        "both_correct": int(paired_counts["a"]),
        "baseline_only_correct": int(paired_counts["b"]),
        "enhanced_only_correct": int(paired_counts["c"]),
        "both_incorrect": int(paired_counts["d"]),
        "discordant_pairs": (
            int(paired_counts["b"])
            + int(paired_counts["c"])
        )
    },
    "mcnemar": {
        "method": mcnemar_result["method"],
        "statistic": mcnemar_result["test_statistic"],
        "p_value": mcnemar_result["p_value_two_sided"],
        "alpha": mcnemar_result["alpha"],
        "reject_null": mcnemar_result[
            "reject_equal_marginal_error_rates"
        ],
        "accuracy_difference": mcnemar_result[
            "accuracy_difference"
        ],
        "accuracy_difference_ci95": mcnemar_result[
            "accuracy_difference_ci95_approximate"
        ]
    },
    "interpretation": {
        "mcnemar_scope": (
            "Paired overall classification correctness"
        ),
        "macro_f1_scope": (
            "Descriptive class-balanced classification performance"
        ),
        "evaluation_limitations": [
            "Potential dependence among related network-flow records",
            "Very small test supports for some attack classes",
            "Successive evaluations using the same CIC test partition"
        ]
    },
    "model_training_performed": False
}

summary_path = (
    OUTPUT_DIR_08 / "RF08_statistical_summary.json"
)

summary_bytes = (
    json.dumps(
        statistical_summary,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

summary_status = write_bytes_once(
    summary_path,
    summary_bytes
)

print("\nSTATISTICAL SUMMARY")
print("Summary JSON:", summary_status)
print(
    "Macro F1 difference:",
    f"{statistical_summary['aggregate_metrics']['macro_f1_difference']:+.6f}"
)
print(
    "McNemar p-value:",
    f"{statistical_summary['mcnemar']['p_value']:.7f}"
)

RF08_EXPORTS_COMPLETE = True

print("\nCELL 11: PASSED")
print("Results directory:", OUTPUT_DIR_08)
print("Model training performed: NO")
print("=" * 72)


CELL 11 — STATISTICAL RESULTS AND FIGURE EXPORT

TABLE EXPORTS
RF08_per_class_comparison.csv : CREATED
RF08_per_class_paired_counts.csv : CREATED
RF08_error_pattern_comparison.csv : CREATED
RF08_baseline_confusion_matrix.csv : CREATED
RF08_enhanced_confusion_matrix.csv : CREATED

FIGURE EXPORTS
Per-class F1 comparison: CREATED
Per-class correctness change: CREATED

STATISTICAL SUMMARY
Summary JSON: CREATED
Macro F1 difference: +0.018702
McNemar p-value: 0.5384046

CELL 11: PASSED
Results directory: /content/drive/MyDrive/MSc_Project/results/RF_STATS_CIC
Model training performed: NO


In [16]:

# ============================================================
# NOTEBOOK 08 — CIC ROBUSTNESS AND STATISTICAL EVALUATION
# CELL 12 — FINAL AUDIT AND COMPLETION RECORD
# ============================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

if not globals().get("RF08_EXPORTS_COMPLETE", False):
    raise RuntimeError("Cell 11 must pass first.")

print("=" * 72)
print("CELL 12 — FINAL AUDIT AND COMPLETION RECORD")
print("=" * 72)

required_files = [
    "RF08_test_true_indices.npy",
    "RF08_baseline_pred_indices.npy",
    "RF08_enhanced_pred_indices.npy",
    "RF08_class_mapping.json",
    "RF08_verified_test_metrics.csv",
    "RF08_paired_contingency.csv",
    "RF08_paired_counts.json",
    "RF08_McNemar_results.json",
    "RF08_McNemar_results.csv",
    "RF08_per_class_comparison.csv",
    "RF08_per_class_paired_counts.csv",
    "RF08_error_pattern_comparison.csv",
    "RF08_baseline_confusion_matrix.csv",
    "RF08_enhanced_confusion_matrix.csv",
    "RF08_per_class_f1_comparison.png",
    "RF08_per_class_correct_change.png",
    "RF08_statistical_summary.json",
]

missing = [
    filename for filename in required_files
    if not (OUTPUT_DIR_08 / filename).is_file()
]

if missing:
    raise RuntimeError(
        "Missing output files: " + ", ".join(missing)
    )

print("\nARTEFACT VERIFICATION")
print("Required files:", len(required_files))
print("Files found:", len(required_files) - len(missing))

y_true = np.load(
    OUTPUT_DIR_08 / "RF08_test_true_indices.npy",
    allow_pickle=False
)

y_b = np.load(
    OUTPUT_DIR_08 / "RF08_baseline_pred_indices.npy",
    allow_pickle=False
)

y_e = np.load(
    OUTPUT_DIR_08 / "RF08_enhanced_pred_indices.npy",
    allow_pickle=False
)

n = len(y_true)

assert n == 378119
assert len(y_b) == len(y_e) == n

bc = y_b == y_true
ec = y_e == y_true

observed = {
    "a": int(np.count_nonzero(bc & ec)),
    "b": int(np.count_nonzero(bc & ~ec)),
    "c": int(np.count_nonzero(~bc & ec)),
    "d": int(np.count_nonzero(~bc & ~ec)),
    "n": n
}

saved_counts = json.loads(
    (
        OUTPUT_DIR_08 / "RF08_paired_counts.json"
    ).read_text(encoding="utf-8")
)

assert observed == saved_counts

saved_test = json.loads(
    (
        OUTPUT_DIR_08 / "RF08_McNemar_results.json"
    ).read_text(encoding="utf-8")
)

assert observed["b"] == saved_test["baseline_only_correct_b"]
assert observed["c"] == saved_test["enhanced_only_correct_c"]
assert observed["a"] == saved_test["both_correct_a"]
assert observed["d"] == saved_test["both_incorrect_d"]

test_result = mcnemar(
    np.array([
        [observed["a"], observed["b"]],
        [observed["c"], observed["d"]]
    ]),
    exact=False,
    correction=True
)

assert np.isclose(
    float(test_result.pvalue),
    float(saved_test["p_value_two_sided"]),
    atol=1e-12,
    rtol=0
)

class_summary = pd.read_csv(
    OUTPUT_DIR_08 / "RF08_per_class_comparison.csv"
)

paired_class = pd.read_csv(
    OUTPUT_DIR_08 / "RF08_per_class_paired_counts.csv"
)

assert len(class_summary) == 15
assert len(paired_class) == 15
assert int(class_summary["Support"].sum()) == n

assert int(
    paired_class["Baseline Only Correct"].sum()
) == observed["b"]

assert int(
    paired_class["Enhanced Only Correct"].sum()
) == observed["c"]

assert int(
    paired_class["Both Incorrect"].sum()
) == observed["d"]

assert int(
    paired_class["Both Correct"].sum()
) == observed["a"]

cm_b = pd.read_csv(
    OUTPUT_DIR_08 / "RF08_baseline_confusion_matrix.csv",
    index_col=0
)

cm_e = pd.read_csv(
    OUTPUT_DIR_08 / "RF08_enhanced_confusion_matrix.csv",
    index_col=0
)

assert cm_b.shape == (15, 15)
assert cm_e.shape == (15, 15)
assert int(cm_b.to_numpy().sum()) == n
assert int(cm_e.to_numpy().sum()) == n

assert int(np.trace(cm_b.to_numpy())) == 377592
assert int(np.trace(cm_e.to_numpy())) == 377602

print("Paired predictions: PASS")
print("Contingency counts: PASS")
print("McNemar result: PASS")
print("Per-class analysis: PASS")
print("Confusion matrices: PASS")

# SHA-256 file fingerprints

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(8 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

artefact_hashes = {
    filename: sha256_file(OUTPUT_DIR_08 / filename)
    for filename in sorted(required_files)
}

source_manifest_path = (
    PROJECT_ROOT / "results/RF_E_CIC/RF_E_CIC_manifest.json"
)

source_manifest = json.loads(
    source_manifest_path.read_text(encoding="utf-8")
)

summary = json.loads(
    (
        OUTPUT_DIR_08 / "RF08_statistical_summary.json"
    ).read_text(encoding="utf-8")
)

assert np.isclose(
    summary["aggregate_metrics"]["macro_f1_difference"],
    0.018702,
    atol=0.000002,
    rtol=0
)

assert np.isclose(
    summary["mcnemar"]["p_value"],
    float(test_result.pvalue),
    atol=1e-12,
    rtol=0
)

completion = {
    "notebook": "08_RF_Robustness_Statistics_CICIDS2017",
    "status": "COMPLETE",
    "dataset": "CIC-IDS2017",
    "baseline_model": "RF-B-CIC",
    "enhanced_model": "RF-E-CIC",
    "test_observations": n,
    "class_count": 15,
    "baseline_correct": int(np.count_nonzero(bc)),
    "enhanced_correct": int(np.count_nonzero(ec)),
    "paired_contingency": observed,
    "mcnemar_method": saved_test["method"],
    "mcnemar_p_value": float(test_result.pvalue),
    "macro_f1_change": summary[
        "aggregate_metrics"
    ]["macro_f1_difference"],
    "source_model_fingerprint": source_manifest[
        "source_savedmodel_sha256_fingerprint"
    ],
    "output_file_hashes": artefact_hashes,
    "model_training_performed": False,
    "next_stage": "09_NSLKDD_Preprocessing"
}

completion_path = (
    OUTPUT_DIR_08 / "RF08_NOTEBOOK_COMPLETE.json"
)

completion_bytes = (
    json.dumps(
        completion,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

completion_status = write_bytes_once(
    completion_path,
    completion_bytes
)

print("\nCOMPLETION RECORD")
print("Status:", completion_status)
print("File:", completion_path)
print("SHA-256 fingerprints:", len(artefact_hashes))
print("Baseline correct:", completion["baseline_correct"])
print("Enhanced correct:", completion["enhanced_correct"])
print(
    "Macro F1 improvement:",
    f"{completion['macro_f1_change']:+.6f}"
)
print(
    "McNemar p-value:",
    f"{completion['mcnemar_p_value']:.7f}"
)

print("\n" + "=" * 72)
print("NOTEBOOK 08: COMPLETE")
print("Model training performed: NO")
print("Next stage: 09_NSLKDD_Preprocessing")
print("=" * 72)


CELL 12 — FINAL AUDIT AND COMPLETION RECORD

ARTEFACT VERIFICATION
Required files: 17
Files found: 17
Paired predictions: PASS
Contingency counts: PASS
McNemar result: PASS
Per-class analysis: PASS
Confusion matrices: PASS

COMPLETION RECORD
Status: CREATED
File: /content/drive/MyDrive/MSc_Project/results/RF_STATS_CIC/RF08_NOTEBOOK_COMPLETE.json
SHA-256 fingerprints: 17
Baseline correct: 377592
Enhanced correct: 377602
Macro F1 improvement: +0.018702
McNemar p-value: 0.5384046

NOTEBOOK 08: COMPLETE
Model training performed: NO
Next stage: 09_NSLKDD_Preprocessing
